# DINOv3 for Plasma Segmentation - Exploration

This notebook explores using DINOv3 (facebook/dinov3-vits16) for object detection and segmentation on plasma data.

DINOv3 is the latest version (released August 2025) with **register tokens** for improved dense prediction tasks.

We'll start by loading a pretrained DINOv3 model and visualizing its feature extraction capabilities without training.

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os

# Add ingestion_program to path (handle both relative and absolute paths)
notebook_dir = Path(os.getcwd())
ingestion_path = notebook_dir / 'ingestion_program'
solution_path = notebook_dir / 'solution'

# Add paths if they exist
if ingestion_path.exists():
    sys.path.insert(0, str(ingestion_path))
if solution_path.exists():
    sys.path.insert(0, str(solution_path))

from transformers import AutoModel, AutoImageProcessor
from tokam2d_utils import TokamDataset
from train_model_dinov3 import DINOv3Segmentation

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Load Pretrained DINOv2 Model

In [ ]:
# Load the pretrained DINOv3 backbone
print("Loading DINOv3 model...")
model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"
dinov3_backbone = AutoModel.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
dinov3_backbone.to(device)
dinov3_backbone.eval()

print(f"Model loaded on {device}")
print(f"Hidden size: {dinov3_backbone.config.hidden_size}")
print(f"Patch size: {dinov3_backbone.config.patch_size}")
print(f"Image size: {dinov3_backbone.config.image_size}")
print(f"Number of register tokens: {dinov3_backbone.config.num_register_tokens}")

## 3. Load Sample Data

Load a sample from the tokamak dataset to see how DINOv2 processes it.

In [ ]:
# Specify your data directory path here
# Example: data_dir = Path('../data/training')
data_dir = Path('./dev_phase/input_data/train')  # Adjust this path as needed

if data_dir.exists():
    dataset = TokamDataset(data_dir, include_unlabeled=True)
    print(f"Dataset loaded with {len(dataset)} samples")
    
    # Get a sample
    sample_image, sample_target = dataset[0]
    print(f"Image shape: {sample_image.shape}")
    print(f"Target keys: {sample_target.keys() if sample_target else 'None'}")
else:
    print(f"Data directory not found: {data_dir}")
    print("Creating synthetic sample for demonstration...")
    sample_image = torch.randn(1, 224, 224)
    sample_target = None

## 4. Visualize Input Image

In [ ]:
# Visualize the sample image
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(sample_image.squeeze().cpu().numpy(), cmap='viridis')
plt.title('Original Image')
plt.colorbar()

# Show normalized version (what DINOv2 will see)
plt.subplot(1, 2, 2)
normalized = F.interpolate(sample_image.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False)
plt.imshow(normalized.squeeze().cpu().numpy(), cmap='viridis')
plt.title('Normalized to 224x224')
plt.colorbar()

plt.tight_layout()
plt.show()

print(f"Image statistics:")
print(f"  Min: {sample_image.min():.4f}")
print(f"  Max: {sample_image.max():.4f}")
print(f"  Mean: {sample_image.mean():.4f}")
print(f"  Std: {sample_image.std():.4f}")

## 5. Extract DINOv2 Features

In [ ]:
# Prepare image for DINOv3 (expects 3-channel RGB, 224x224)
# Plasma data is single-channel, so we convert it to 3 channels

# Add batch dimension if needed
if sample_image.dim() == 2:  # (H, W)
    image_input = sample_image.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
elif sample_image.dim() == 3:  # (C, H, W)
    image_input = sample_image.unsqueeze(0)  # (1, C, H, W)
else:
    image_input = sample_image

# Resize to 224x224
image_input = F.interpolate(
    image_input,
    size=(224, 224),
    mode='bilinear',
    align_corners=False
)

# Convert single channel to 3 channels (grayscale to RGB)
if image_input.shape[1] == 1:
    image_input = image_input.repeat(1, 3, 1, 1)  # (1, 3, 224, 224)

image_input = image_input.to(device)

print(f"Input image shape: {image_input.shape} (should be [1, 3, 224, 224])")

# Extract features following HuggingFace documentation
batch_size, _, img_height, img_width = image_input.shape
patch_size = dinov3_backbone.config.patch_size
num_patches_height = img_height // patch_size
num_patches_width = img_width // patch_size
num_patches_flat = num_patches_height * num_patches_width

with torch.inference_mode():
    outputs = dinov3_backbone(image_input)
    last_hidden_states = outputs.last_hidden_state

print(f"\nFeature shape: {last_hidden_states.shape}")
print(f"  Expected: [{batch_size}, {1 + dinov3_backbone.config.num_register_tokens + num_patches_flat}, {dinov3_backbone.config.hidden_size}]")
print(f"  - 1 CLS token")
print(f"  - {dinov3_backbone.config.num_register_tokens} register tokens")
print(f"  - {num_patches_flat} patch tokens ({num_patches_height}x{num_patches_width})")
print(f"  - {dinov3_backbone.config.hidden_size} dimensional features")

# Extract tokens following HuggingFace pattern
cls_token = last_hidden_states[:, 0, :]
patch_features_flat = last_hidden_states[:, 1 + dinov3_backbone.config.num_register_tokens:, :]
patch_features = patch_features_flat.unflatten(1, (num_patches_height, num_patches_width))

print(f"\nCLS token shape: {cls_token.shape}")
print(f"Patch features shape: {patch_features.shape}")

# Reshape to (B, C, H, W) for visualization
spatial_features = patch_features.permute(0, 3, 1, 2)
patch_size_grid = num_patches_height  # Store for later use
print(f"Spatial features shape: {spatial_features.shape} (B, C, H, W)")

## 5.1. Visualize Patch Token Embeddings

Each 16x16 patch gets its own 384-dimensional embedding. Let's visualize these local embeddings and see how they correspond to spatial locations in the image.

In [ ]:
# Visualize patch tokens structure
print("=" * 60)
print("PATCH TOKEN EMBEDDINGS - Local features for dense tasks")
print("=" * 60)

print(f"\n📍 Spatial Structure:")
print(f"   Image size: 224 x 224 pixels")
print(f"   Patch size: {dinov3_backbone.config.patch_size} x {dinov3_backbone.config.patch_size} pixels")
print(f"   Grid: {num_patches_height} x {num_patches_width} patches")
print(f"   Total patches: {num_patches_flat}")

print(f"\n🔢 Token Dimensions:")
print(f"   patch_features_flat shape: {patch_features_flat.shape}")
print(f"   → (batch=1, num_patches={num_patches_flat}, embedding_dim=384)")
print(f"\n   patch_features (unflattened) shape: {patch_features.shape}")
print(f"   → (batch=1, height={num_patches_height}, width={num_patches_width}, embedding_dim=384)")

print(f"\n🎯 Usage for Dense Prediction:")
print(f"   Each of the {num_patches_flat} patches has its own 384-D embedding")
print(f"   These local features preserve spatial structure")
print(f"   Perfect for pixel-wise tasks like segmentation!")

# Visualize individual patch embeddings
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

# Select 10 random patch locations
random_patches = np.random.choice(num_patches_flat, 10, replace=False)
patch_coords = [(idx // num_patches_width, idx % num_patches_width) for idx in random_patches]

for plot_idx, (h, w) in enumerate(patch_coords):
    ax = axes[plot_idx // 5, plot_idx % 5]
    
    # Get the 384-dimensional embedding for this patch
    patch_embedding = patch_features[0, h, w, :].cpu().numpy()  # (384,)
    
    # Visualize the embedding as a 1D signal
    ax.plot(patch_embedding, linewidth=0.5, alpha=0.7)
    ax.set_title(f'Patch [{h},{w}]', fontsize=10)
    ax.set_xlabel('Embedding dimension', fontsize=8)
    ax.set_ylabel('Value', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=7)
    
    # Add statistics
    stats_text = f'μ={patch_embedding.mean():.2f}\nσ={patch_embedding.std():.2f}'
    ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, 
            fontsize=7, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Individual Patch Token Embeddings (384-D vectors)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Show spatial correspondence
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Prepare sample image for visualization (handle different input shapes)
sample_viz = sample_image.squeeze().cpu().numpy()  # Remove extra dims

# Original image
axes[0].imshow(sample_viz, cmap='viridis')
axes[0].set_title(f'Original Plasma Image\n({sample_viz.shape[0]}x{sample_viz.shape[1]} pixels)', 
                  fontsize=12, fontweight='bold')
axes[0].axis('off')

# Patch grid overlay (on 224x224 normalized version)
# Resize to 224x224 for grid overlay
if sample_image.dim() == 2:
    img_224 = sample_image.unsqueeze(0).unsqueeze(0)
elif sample_image.dim() == 3:
    img_224 = sample_image.unsqueeze(0)
else:
    img_224 = sample_image

img_224 = F.interpolate(img_224, size=(224, 224), mode='bilinear', align_corners=False)
img_224_viz = img_224.squeeze().cpu().numpy()

axes[1].imshow(img_224_viz, cmap='viridis', alpha=0.7)
# Draw grid lines
for i in range(num_patches_height + 1):
    axes[1].axhline(i * dinov3_backbone.config.patch_size - 0.5, color='red', linewidth=1, alpha=0.8)
for j in range(num_patches_width + 1):
    axes[1].axvline(j * dinov3_backbone.config.patch_size - 0.5, color='red', linewidth=1, alpha=0.8)
axes[1].set_title(f'Patch Grid\n({num_patches_height}x{num_patches_width} patches of 16x16 pixels)', 
                  fontsize=12, fontweight='bold')
axes[1].set_xlim(-0.5, 224 - 0.5)
axes[1].set_ylim(224 - 0.5, -0.5)
axes[1].axis('off')

# Patch embeddings spatial layout (show embedding norms)
patch_norms = torch.norm(patch_features[0], dim=-1).cpu().numpy()  # (14, 14)
im = axes[2].imshow(patch_norms, cmap='plasma', interpolation='nearest')
axes[2].set_title('Patch Embedding Magnitudes\n(||embedding||₂ for each patch)', 
                  fontsize=12, fontweight='bold')
axes[2].set_xlabel('Patch X coordinate', fontsize=10)
axes[2].set_ylabel('Patch Y coordinate', fontsize=10)
plt.colorbar(im, ax=axes[2], label='L2 norm')

# Add grid
for i in range(num_patches_height + 1):
    axes[2].axhline(i - 0.5, color='white', linewidth=0.5, alpha=0.3)
for j in range(num_patches_width + 1):
    axes[2].axvline(j - 0.5, color='white', linewidth=0.5, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ These {num_patches_flat} patch tokens (each 384-D) form the basis for dense prediction!")
print(f"✓ Register tokens keep global info separate, so these stay clean and spatially meaningful")

## 6. Visualize Feature Maps

In [ ]:
# Visualize some feature channels
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# Select 8 random feature channels to visualize
num_features = spatial_features.shape[1]
selected_channels = np.random.choice(num_features, 8, replace=False)

for idx, channel in enumerate(selected_channels):
    feature_map = spatial_features[0, channel].cpu().numpy()
    axes[idx].imshow(feature_map, cmap='viridis')
    axes[idx].set_title(f'Feature Channel {channel}')
    axes[idx].axis('off')

plt.suptitle('DINOv2 Feature Maps (Random Channels)', fontsize=16)
plt.tight_layout()
plt.show()

## 7. Visualize Attention Maps

DINOv2 uses self-attention. We can visualize attention patterns to see what the model focuses on.

In [ ]:
# Get attention weights from the model following HuggingFace pattern
# Note: DINOv3 model needs to explicitly request attentions
with torch.inference_mode():
    outputs = dinov3_backbone(image_input, output_attentions=True)

print(f"Available output keys: {outputs.keys()}")

# According to HF docs, attentions are returned when output_attentions=True
if hasattr(outputs, 'attentions') and outputs.attentions is not None:
    attentions = outputs.attentions
    print(f"\n✓ Attention weights available!")
    print(f"Number of layers: {len(attentions)}")
    print(f"Attention shape (last layer): {attentions[-1].shape}")
    print(f"  Format: (batch, num_heads, num_tokens, num_tokens)")
    
    # Get attention from CLS token to all patches in the last layer
    # Shape: (batch, num_heads, num_tokens, num_tokens)
    last_layer_attn = attentions[-1]  # Last layer
    
    # Extract CLS token attention to patches
    # Token order: [CLS, register_tokens..., patch_tokens...]
    num_register = dinov3_backbone.config.num_register_tokens
    
    # CLS attention to all patches (skip CLS and register tokens)
    cls_to_patches = last_layer_attn[0, :, 0, 1 + num_register:]  # (num_heads, num_patches)
    
    # Average over attention heads
    avg_attention = cls_to_patches.mean(dim=0)  # (num_patches,)
    
    # Reshape to 2D grid
    attention_map = avg_attention.reshape(patch_size_grid, patch_size_grid).cpu().numpy()
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(sample_image.squeeze().cpu().numpy(), cmap='viridis')
    axes[0].set_title('Original Plasma Density')
    axes[0].axis('off')
    
    axes[1].imshow(attention_map, cmap='hot', interpolation='bilinear')
    axes[1].set_title('CLS Attention to Patches')
    axes[1].axis('off')
    
    # Overlay - properly handle sample_image dimensions
    if sample_image.dim() == 2:
        img_for_resize = sample_image.unsqueeze(0).unsqueeze(0)
    elif sample_image.dim() == 3:
        img_for_resize = sample_image.unsqueeze(0)
    else:
        img_for_resize = sample_image
    
    img_resized = F.interpolate(
        img_for_resize,
        size=(patch_size_grid, patch_size_grid),
        mode='bilinear',
        align_corners=False
    ).squeeze().cpu().numpy()
    
    axes[2].imshow(img_resized, cmap='gray', alpha=0.6)
    im = axes[2].imshow(attention_map, cmap='hot', alpha=0.4, interpolation='bilinear')
    axes[2].set_title('Attention Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("\n⚠️  Attention weights not returned by model")
    print("Using feature activation as alternative visualization\n")
    
    # Alternative: visualize feature norms
    feature_activation = torch.norm(spatial_features[0], dim=0).cpu().numpy()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(sample_image.squeeze().cpu().numpy(), cmap='viridis')
    axes[0].set_title('Original Plasma Density')
    axes[0].axis('off')
    
    axes[1].imshow(feature_activation, cmap='hot', interpolation='bilinear')
    axes[1].set_title('Feature Activation Map')
    axes[1].axis('off')
    
    # Overlay - properly handle sample_image dimensions
    if sample_image.dim() == 2:
        img_for_resize = sample_image.unsqueeze(0).unsqueeze(0)
    elif sample_image.dim() == 3:
        img_for_resize = sample_image.unsqueeze(0)
    else:
        img_for_resize = sample_image
    
    img_resized = F.interpolate(
        img_for_resize,
        size=feature_activation.shape,
        mode='bilinear',
        align_corners=False
    ).squeeze().cpu().numpy()
    
    axes[2].imshow(img_resized, cmap='gray', alpha=0.6)
    axes[2].imshow(feature_activation, cmap='hot', alpha=0.4, interpolation='bilinear')
    axes[2].set_title('Activation Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

## 8. Load Full DINOv2 Segmentation Model

Now let's load our custom DINOv2Segmentation model with detection and segmentation heads.

In [ ]:
# Load the full segmentation model
model = DINOv3Segmentation(num_classes=2, pretrained=True, model_name=model_name)
model.to(device)
model.eval()

print("DINOv3Segmentation model loaded")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 9. Test Object Detection (Pretrained)

Let's see how the model performs with randomly initialized detection heads (before training).

In [ ]:
# Run inference
with torch.no_grad():
    predictions = model([sample_image.to(device)])

pred = predictions[0]
print(f"Predictions:")
print(f"  Boxes: {pred['boxes'].shape}")
print(f"  Labels: {pred['labels'].shape}")
print(f"  Scores: {pred['scores'].shape}")
print(f"  Masks: {pred['masks'].shape}")

if len(pred['boxes']) > 0:
    print(f"\nDetected {len(pred['boxes'])} objects:")
    for i in range(len(pred['boxes'])):
        print(f"  Object {i+1}: Label={pred['labels'][i].item()}, Score={pred['scores'][i].item():.3f}")
else:
    print("\nNo objects detected (expected for untrained model)")

## 10. Visualize Predictions

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original image
axes[0].imshow(sample_image.squeeze().cpu().numpy(), cmap='viridis')
axes[0].set_title('Original Image')
axes[0].axis('off')

# Predicted segmentation mask (class 1)
if pred['masks'].shape[0] > 1:
    mask = torch.sigmoid(pred['masks'][1]).cpu().numpy()
    axes[1].imshow(mask, cmap='hot')
    axes[1].set_title('Predicted Mask (Class 1 - Plasma)')
    axes[1].axis('off')
else:
    axes[1].text(0.5, 0.5, 'No mask available', ha='center', va='center')
    axes[1].axis('off')

# Image with bounding boxes
axes[2].imshow(sample_image.squeeze().cpu().numpy(), cmap='viridis')
if len(pred['boxes']) > 0:
    for box, label, score in zip(pred['boxes'], pred['labels'], pred['scores']):
        x, y, w, h = box.cpu().numpy()
        # Convert from center format to corner format if needed
        rect = plt.Rectangle(
            (x - w/2, y - h/2), w, h,
            fill=False, edgecolor='red', linewidth=2
        )
        axes[2].add_patch(rect)
        axes[2].text(
            x, y - h/2 - 5,
            f'L{label.item()}: {score.item():.2f}',
            color='red', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )
axes[2].set_title('Predicted Bounding Boxes')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 11. Feature Analysis

Let's analyze the features extracted by DINOv2 to understand what it captures.

In [ ]:
# Compute feature statistics
with torch.inference_mode():
    # Get features from backbone
    backbone_outputs = model.backbone(image_input)
    last_hidden_states = backbone_outputs.last_hidden_state
    
    # Get patch features (skip CLS and register tokens) following HF pattern
    num_register = model.num_register_tokens
    patch_features_flat = last_hidden_states[:, 1 + num_register:, :]
    
    # Compute statistics
    feature_norms = torch.norm(patch_features_flat, dim=-1)
    feature_mean = patch_features_flat.mean(dim=-1)
    feature_std = patch_features_flat.std(dim=-1)

# Visualize feature statistics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, data, title in zip(
    axes,
    [feature_norms, feature_mean, feature_std],
    ['Feature Norms', 'Feature Mean', 'Feature Std']
):
    stat_map = data.reshape(patch_size_grid, patch_size_grid).cpu().numpy()
    im = ax.imshow(stat_map, cmap='viridis')
    ax.set_title(title)
    ax.axis('off')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print(f"\nFeature statistics:")
print(f"  Norm - min: {feature_norms.min():.4f}, max: {feature_norms.max():.4f}, mean: {feature_norms.mean():.4f}")
print(f"  Mean - min: {feature_mean.min():.4f}, max: {feature_mean.max():.4f}, mean: {feature_mean.mean():.4f}")
print(f"  Std - min: {feature_std.min():.4f}, max: {feature_std.max():.4f}, mean: {feature_std.mean():.4f}")

## 12. Summary

In this notebook, we explored:

1. ✅ Loading a pretrained DINOv3 model (facebook/dinov3-vits16-pretrain-lvd1689m)
2. ✅ Understanding DINOv3's register tokens for better dense predictions
3. ✅ Extracting and visualizing features from plasma images
4. ✅ Analyzing attention patterns from the vision transformer
5. ✅ Testing our custom DINOv3Segmentation model (untrained)
6. ✅ Visualizing predictions from randomly initialized heads

### Key DINOv3 Improvements over DINOv2:

- **Register tokens**: Dedicated memory slots for global information
- **Cleaner attention maps**: Reduced high-norm artifacts in patch tokens
- **Better dense prediction**: Improved performance on segmentation tasks
- **Released August 2025**: Latest state-of-the-art foundation model

### Next Steps:

- Train the model using `train_model()` function from `train_model_dinov3.py`
- Fine-tune the backbone by calling `model.unfreeze_backbone()`
- Experiment with different learning rates and training strategies
- Try larger models: `facebook/dinov3-vitb16-pretrain-lvd1689m` or `facebook/dinov3-vit7b16-pretrain-lvd1689m`
- Evaluate on validation data and visualize trained predictions

The pretrained DINOv3 backbone with register tokens provides even stronger visual features for plasma segmentation!